# Fake News Classification on Big Kaggle WELFake Dataset (72,134 Articles)
### End-to-End DistilBERT Fine-Tuning with Kaggle GPU Acceleration vs. Baseline

This notebook trains on **WELFake** (Word Embedding-based Labeled Fake news Dataset), which contains **72,134 articles** merged from 4 major news benchmarks (Kaggle Fake News, McIntire, Reuters, BuzzFeed).

**Features**:
1. **Big Dataset (72k)**: Solves the single-source 'Reuters' bias by blending multiple news sources.
2. **Mixed Precision (FP16)**: Fast GPU execution on Kaggle Tesla T4 / P100.
3. **Dual Model Evaluation**: Compares fine-tuned DistilBERT against Sublinear TF-IDF + Logistic Regression.
4. **Adversarial Probing & Error Analysis**: Tests stylistic confounding vs factual truth.

In [ ]:
# Install dependencies if not already present
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas

In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'Active Device: {device}')
if device.type == 'cuda':
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 1. Load Big Kaggle WELFake Dataset (72,134 rows)

In [ ]:
# Load either from Kaggle input path or from Hugging Face hub mirror
kaggle_csv_path = '/kaggle/input/fake-news-classification/WELFake_Dataset.csv'

if os.path.exists(kaggle_csv_path):
    print(f'Loading from Kaggle filesystem: {kaggle_csv_path}')
    df = pd.read_csv(kaggle_csv_path)
else:
    print('Loading from Hugging Face mirror of Kaggle WELFake...')
    ds = load_dataset('davanstrien/WELFake')
    df = pd.DataFrame(ds['train'])

# Clean text
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['full_text'] = df['title'] + ' -- ' + df['text']
df = df[df['full_text'].str.strip() != ''].reset_index(drop=True)

print(f'Total Cleaned Articles: {len(df):,}')
print(f'Class Distribution: Real News (1) = {sum(df["label"]==1):,}, Fake News (0) = {sum(df["label"]==0):,}')

## 2. Stratified Train / Test Partitioning

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['label'], random_state=42
)
print(f'Training Set: {len(train_df):,} articles')
print(f'Testing Set:  {len(test_df):,} articles')

## 3. Fast Baseline: Sublinear TF-IDF (25k Features) + Logistic Regression

In [ ]:
print('Extracting TF-IDF n-grams from training corpus...')
t0 = time.time()
vectorizer = TfidfVectorizer(
    max_features=25000, 
    ngram_range=(1, 2), 
    sublinear_tf=True, 
    stop_words='english'
)
X_train = vectorizer.fit_transform(train_df['full_text'])
X_test = vectorizer.transform(test_df['full_text'])

clf = LogisticRegression(C=2.0, max_iter=1000, random_state=42)
clf.fit(X_train, train_df['label'].values)

lr_preds = clf.predict(X_test)
lr_probs = clf.predict_proba(X_test)[:, 1]

lr_acc = accuracy_score(test_df['label'].values, lr_preds)
lr_f1 = f1_score(test_df['label'].values, lr_preds, average='macro')
lr_auc = roc_auc_score(test_df['label'].values, lr_probs)

print(f'Logistic Regression Baseline Complete in {time.time()-t0:.1f}s:')
print(f'  • Accuracy: {lr_acc*100:.2f}%')
print(f'  • Macro F1: {lr_f1:.4f}')
print(f'  • ROC-AUC:  {lr_auc:.4f}')

## 4. Fine-Tuning DistilBERT with PyTorch & Mixed Precision

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=192):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# On Kaggle GPU, we can fine-tune on a large sample (e.g., 20,000 articles) or full set
db_train = train_df.sample(n=20000, random_state=42)
db_test = test_df.sample(n=3000, random_state=42)

train_loader = DataLoader(NewsDataset(db_train['full_text'], db_train['label'], tokenizer), batch_size=32, shuffle=True)
test_loader = DataLoader(NewsDataset(db_test['full_text'], db_test['label'], tokenizer), batch_size=32, shuffle=False)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

print('Starting DistilBERT Training on 20,000 WELFake articles...')

In [ ]:
epochs = 2

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}')
    
    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{running_loss / (pbar.n + 1):.4f}'})

print('Training complete! Evaluating on test set...')

## 5. Evaluation & Comparative Metrics

In [ ]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = np.argmax(logits.cpu().numpy(), axis=1)
        
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(batch['labels'].numpy())

db_acc = accuracy_score(all_labels, all_preds)
db_f1 = f1_score(all_labels, all_preds, average='macro')
db_auc = roc_auc_score(all_labels, all_probs)

print('=' * 65)
print('              FINAL COMPARATIVE BENCHMARK (WELFAKE 72K)          ')
print('=' * 65)
print(f'Logistic Regression : {lr_acc*100:.2f}% Acc | {lr_f1:.4f} F1 | {lr_auc:.4f} AUC')
print(f'Fine-Tuned DistilBERT: {db_acc*100:.2f}% Acc | {db_f1:.4f} F1 | {db_auc:.4f} AUC')
print('=' * 65)

## 6. Adversarial Probing: Ground Truth vs. Linguistic Style

In [ ]:
probes = [
    {
        'desc': 'Falsehood in Reuters Wire Style',
        'text': 'WASHINGTON (Reuters) - The International Monetary Fund announced on Wednesday that the Moon\'s core consists predominantly of concentrated dairy proteins.'
    },
    {
        'desc': 'Verified Fact in Tabloid Style',
        'text': 'OMG YOU WON\'T BELIEVE THIS!! A METEOR EXPLODED OVER RUSSIA INJURING 1000+ PEOPLE AND BLOWING OUT WINDOWS! WATCH NOW BEFORE DELETED!'
    }
]

print('\nAdversarial Probe Predictions on DistilBERT:')
for p in probes:
    enc = tokenizer(p['text'], return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=1)[0].cpu().numpy()
    pred = 'REAL' if probs[1] > 0.5 else 'FAKE'
    print(f"\n[{p['desc']}]")
    print(f"  Text: {p['text']}")
    print(f"  -> Model Output: {pred} with {max(probs)*100:.2f}% confidence")